# Domain E: Functional Connectivity and Circuit Interactions

This notebook addresses five questions:
- **E1:** Do noise correlations reveal cell-type-specific connectivity motifs?
- **E2:** Can transfer entropy / Granger causality reveal directed functional interactions?
- **E3:** How does population coupling relate to cell type and spatial position?
- **E4:** Does spontaneous activity structure during grey-screen epochs reveal cell-type-specific network dynamics? *(Zarr data)*
- **E5:** Do catch-trial responses reveal cell-type-specific expectation signals? *(Zarr data)*

**Data:** Zarr multimodal stores with ΔF/F traces (cells × trials), 3D coordinates, cell-type labels, gene expression, spontaneous activity, catch trials, and GLM data.

**Note:** E2 (Granger causality) requires continuous time-series data. The current dataset stores trial-level responses. Sections requiring continuous ΔF/F traces are marked `# ⚠️ REQUIRES: continuous time-series data`.

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.stats import kruskal, spearmanr, pearsonr, mannwhitneyu
from scipy.spatial.distance import cdist, pdist, squareform
import warnings
warnings.filterwarnings('ignore')

from functions.data_loading import load_mouse_zarr, load_zarr_10hz
from functions.analysis import xcorr_pair, pairwise_granger

sns.set_context('talk')
sns.set_style('whitegrid')

# ══════════════════════════════════════════════════════════════════════
# Load data from zarr multimodal stores
# ══════════════════════════════════════════════════════════════════════
ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

adata_list = [load_mouse_zarr(mid, zarr_dir=ZARR_DIR, include_genes=True) for mid in MOUSE_IDS]

obs = pd.concat([a.obs for a in adata_list], ignore_index=True)
X = np.vstack([a.X for a in adata_list])
var = adata_list[0].var.copy()

SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]

# Backward-compatible adata object for downstream cells
adata = SimpleNamespace(X=X, obs=obs, var=var, n_obs=len(obs), n_vars=X.shape[1])

coords = obs[['x_loc', 'y_loc', 'z_loc']].values.astype(float)
orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])

# Gene columns
META_COLS = {'unique_id', 'mouse_id', 'class_name', 'subclass_name', 'subclass_label',
             'supertype_name', 'cluster_name', 'cluster_alias', 'x_loc', 'y_loc', 'z_loc'}
GENE_COLS = [c for c in obs.columns if c not in META_COLS]

print(f"Total cells: {len(obs)}, Trials: {X.shape[1]}, Genes: {len(GENE_COLS)}")

## E1: Noise Correlations and Cell-Type-Specific Connectivity Motifs

Compute trial-by-trial noise correlations (residual correlations after removing stimulus-driven component). Build subclass × subclass average noise correlation matrix. Relate to spatial distance and gene expression similarity.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# E1.1  Compute noise correlations per mouse
# ══════════════════════════════════════════════════════════════════════

# For each unique stimulus condition, subtract the trial-averaged response,
# then compute pairwise Pearson correlation of the residuals.

# Process one mouse at a time to keep matrices manageable
mouse_ids = obs['mouse_id'].unique()
contrast_blocks = var['stim_block'].isin([0.0, 2.0])

noise_corr_per_mouse = {}
signal_corr_per_mouse = {}

for mouse in mouse_ids:
    m_mask = obs['mouse_id'].values == mouse
    m_X = X[m_mask]
    n_m = m_mask.sum()
    
    # Collect residuals across all stimulus conditions
    residuals_list = []
    signal_vectors = []
    
    for ori in orientations:
        for contrast in [0.05, 0.1, 0.2, 0.4, 0.8]:
            mask = contrast_blocks & (var['orientation'] == ori) & (var['contrast'] == contrast)
            trial_idx = np.where(mask.values)[0]
            if len(trial_idx) >= 3:
                trial_resp = m_X[:, trial_idx]  # cells x trials
                mean_resp = np.nanmean(trial_resp, axis=1)
                signal_vectors.append(mean_resp)
                residuals = trial_resp - mean_resp[:, np.newaxis]
                residuals_list.append(residuals)
    
    if residuals_list:
        all_residuals = np.hstack(residuals_list)  # cells x total_residual_trials
        nc = np.corrcoef(all_residuals)
        noise_corr_per_mouse[mouse] = nc
        
        signal_mat = np.column_stack(signal_vectors)  # cells x conditions
        sc = np.corrcoef(signal_mat)
        signal_corr_per_mouse[mouse] = sc
        
        print(f"Mouse {mouse}: {n_m} cells, {all_residuals.shape[1]} residual trials, "
              f"mean noise corr = {nc[np.triu_indices(n_m, k=1)].mean():.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# E1.2  Subclass × Subclass noise correlation matrix
# ══════════════════════════════════════════════════════════════════════

n_sc = len(present_subclasses)
nc_subclass_matrix = np.zeros((n_sc, n_sc))
nc_subclass_counts = np.zeros((n_sc, n_sc))

for mouse in mouse_ids:
    m_mask = obs['mouse_id'].values == mouse
    m_subclass = obs['subclass_name'].values[m_mask]
    nc = noise_corr_per_mouse.get(mouse)
    if nc is None:
        continue
    
    n_m = m_mask.sum()
    for si, sc1 in enumerate(present_subclasses):
        for sj, sc2 in enumerate(present_subclasses):
            mask1 = m_subclass == sc1
            mask2 = m_subclass == sc2
            if mask1.sum() == 0 or mask2.sum() == 0:
                continue
            
            nc_sub = nc[np.ix_(np.where(mask1)[0], np.where(mask2)[0])]
            if si == sj:
                # Same subclass: use upper triangle only
                triu = np.triu_indices(nc_sub.shape[0], k=1)
                if len(triu[0]) > 0:
                    vals = nc_sub[triu]
                    nc_subclass_matrix[si, sj] += np.nansum(vals)
                    nc_subclass_counts[si, sj] += np.sum(~np.isnan(vals))
            else:
                vals = nc_sub.flatten()
                nc_subclass_matrix[si, sj] += np.nansum(vals)
                nc_subclass_counts[si, sj] += np.sum(~np.isnan(vals))

nc_subclass_avg = nc_subclass_matrix / np.maximum(nc_subclass_counts, 1)

# ── Visualization: Subclass × Subclass noise correlation heatmap ──
short_labels = [SUBCLASS_SHORT[s] for s in present_subclasses]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
sns.heatmap(nc_subclass_avg, annot=True, fmt='.4f', xticklabels=short_labels,
            yticklabels=short_labels, cmap='RdBu_r', center=0, ax=ax)
ax.set_title('E1: Mean Noise Correlation (Subclass × Subclass)', fontweight='bold')

# Same-type vs cross-type comparison
ax = axes[1]
same_type_nc = []
cross_type_nc = []
for mouse in mouse_ids:
    m_mask = obs['mouse_id'].values == mouse
    m_subclass = obs['subclass_name'].values[m_mask]
    nc = noise_corr_per_mouse.get(mouse)
    if nc is None:
        continue
    n_m = m_mask.sum()
    triu_i, triu_j = np.triu_indices(n_m, k=1)
    for ii, jj in zip(triu_i, triu_j):
        val = nc[ii, jj]
        if np.isnan(val):
            continue
        if m_subclass[ii] == m_subclass[jj]:
            same_type_nc.append(val)
        else:
            cross_type_nc.append(val)

data_plot = pd.DataFrame({
    'Noise Correlation': same_type_nc + cross_type_nc,
    'Pair Type': ['Same subclass']*len(same_type_nc) + ['Different subclass']*len(cross_type_nc)
})
sns.violinplot(data=data_plot, x='Pair Type', y='Noise Correlation', cut=0, inner='box',
               palette=['red', 'blue'], ax=ax)
ax.axhline(0, color='k', ls='--', alpha=0.4)
ax.set_title('E1: Same-Type vs Cross-Type Noise Correlation', fontweight='bold')
stat, p = mannwhitneyu(same_type_nc, cross_type_nc, alternative='two-sided')
ax.text(0.5, 0.95, f'Mann-Whitney p={p:.2e}', transform=ax.transAxes, ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print(f"Same-type noise corr: mean={np.mean(same_type_nc):.4f}, n={len(same_type_nc)}")
print(f"Cross-type noise corr: mean={np.mean(cross_type_nc):.4f}, n={len(cross_type_nc)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# E1.3  Noise correlations vs distance and gene-expression similarity
#       Mantel test: NC matrix ~ distance matrix + gene similarity
# ══════════════════════════════════════════════════════════════════════

# Use largest mouse for detailed analysis
mouse_pick = mouse_ids[np.argmax([np.sum(obs['mouse_id'].values == m) for m in mouse_ids])]
m_mask = obs['mouse_id'].values == mouse_pick
m_coords = coords[m_mask]
nc = noise_corr_per_mouse[mouse_pick]
n_m = m_mask.sum()

triu_i, triu_j = np.triu_indices(n_m, k=1)
pair_nc = nc[triu_i, triu_j]
pair_dist = squareform(pdist(m_coords))[triu_i, triu_j]

# Gene expression similarity (correlation)
gene_expr = obs.loc[m_mask, GENE_COLS].values.astype(float)
gene_expr = np.nan_to_num(gene_expr, nan=0.0)
nonzero = np.std(gene_expr, axis=0) > 1e-6
gene_corr_mat = np.corrcoef(gene_expr[:, nonzero])
pair_gene_sim = gene_corr_mat[triu_i, triu_j]

# ── Noise correlation vs distance ──
distance_edges = np.arange(0, 400, 25)
distance_centers = (distance_edges[:-1] + distance_edges[1:]) / 2

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# NC vs distance
ax = axes[0]
binned_nc = []
for b in range(len(distance_edges)-1):
    bm = (pair_dist >= distance_edges[b]) & (pair_dist < distance_edges[b+1]) & ~np.isnan(pair_nc)
    if bm.sum() > 10:
        binned_nc.append(np.mean(pair_nc[bm]))
    else:
        binned_nc.append(np.nan)
ax.plot(distance_centers, binned_nc, 'ko-', linewidth=2)
ax.set_xlabel('Pairwise Distance (µm)')
ax.set_ylabel('Mean Noise Correlation')
ax.set_title('E1: Noise Corr vs Distance', fontweight='bold')
ax.axhline(0, color='gray', ls='--', alpha=0.4)

# NC vs gene expression similarity
ax = axes[1]
gene_bins = np.percentile(pair_gene_sim[~np.isnan(pair_gene_sim)], np.arange(0, 101, 10))
gene_centers = (gene_bins[:-1] + gene_bins[1:]) / 2
binned_nc_gene = []
for b in range(len(gene_bins)-1):
    bm = (pair_gene_sim >= gene_bins[b]) & (pair_gene_sim < gene_bins[b+1]) & ~np.isnan(pair_nc)
    if bm.sum() > 10:
        binned_nc_gene.append(np.mean(pair_nc[bm]))
    else:
        binned_nc_gene.append(np.nan)
ax.plot(gene_centers, binned_nc_gene, 'ro-', linewidth=2)
ax.set_xlabel('Gene Expression Similarity (Pearson r)')
ax.set_ylabel('Mean Noise Correlation')
ax.set_title('E1: Noise Corr vs Gene Similarity', fontweight='bold')

# Partial correlation: NC ~ distance + gene_sim + cell-type match
ax = axes[2]
m_subclass = obs['subclass_name'].values[m_mask]
pair_same = (m_subclass[triu_i] == m_subclass[triu_j]).astype(float)

# Simple multiple regression
from numpy.linalg import lstsq
valid = ~np.isnan(pair_nc) & ~np.isnan(pair_gene_sim) & ~np.isnan(pair_dist)
A = np.column_stack([pair_dist[valid], pair_gene_sim[valid], pair_same[valid], np.ones(valid.sum())])
Y = pair_nc[valid]
beta, residuals, _, _ = lstsq(A, Y, rcond=None)
print(f"Multiple regression: NC ~ distance + gene_sim + same_type")
print(f"  β_distance = {beta[0]:.6f}")
print(f"  β_gene_sim = {beta[1]:.4f}")
print(f"  β_same_type = {beta[2]:.4f}")
print(f"  β_intercept = {beta[3]:.4f}")
ss_res = np.sum((Y - A @ beta)**2)
ss_tot = np.sum((Y - np.mean(Y))**2)
r2 = 1 - ss_res / ss_tot
print(f"  R² = {r2:.4f}")

# Bar plot of standardized coefficients
A_std = (A[:, :3] - A[:, :3].mean(axis=0)) / (A[:, :3].std(axis=0) + 1e-8)
A_std_full = np.column_stack([A_std, np.ones(len(A_std))])
beta_std, _, _, _ = lstsq(A_std_full, (Y - Y.mean()) / Y.std(), rcond=None)
ax.bar(['Distance', 'Gene Sim', 'Same Type'], beta_std[:3],
       color=['steelblue', 'salmon', 'green'])
ax.set_ylabel('Standardized β')
ax.set_title(f'E1: Predictors of NC (R²={r2:.3f})', fontweight='bold')
ax.axhline(0, color='k', ls='--', alpha=0.4)

plt.tight_layout()
plt.show()